# MedLens — Data Rebuild (Kaggle / vLLM)

Same pipeline as `medlens_data_rebuild.ipynb` but runs on Kaggle with vLLM on 2× T4 GPUs.

**Pipeline stages:**
1. Install deps + load model
2. Load raw `medlens_train.jsonl` from Kaggle input dataset
3. Profile
4. Dedupe exact (user, asst) pairs
5. Cap per drug-combo
6. Cap per template skeleton
7. Balance severity + split train/eval
8. Paraphrase train via vLLM (batched, checkpointed)
9. Write final jsonl to `/kaggle/working/`
10. Post-write diagnostics

> **Input:** Add your dataset containing `medlens_train.jsonl` as a Kaggle dataset input.  
> **Output:** `/kaggle/working/vtrain.jsonl`, `/kaggle/working/veval.jsonl`

In [ ]:
# Install vLLM and deps
!pip install -q -U vllm accelerate bitsandbytes

## 1. Config

In [ ]:
from pathlib import Path

# ── Input: point at your Kaggle dataset that contains medlens_train.jsonl ──
# Adjust the dataset slug to match what you uploaded.
RAW_JSONL = Path('/kaggle/input/medlens-dataset/medlens_train.jsonl')

# ── Output: all intermediate + final files go here ──
WORK_DIR  = Path('/kaggle/working')
CKPT_DIR  = WORK_DIR / 'rebuild_ckpts'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

OUT_TRAIN = WORK_DIR / 'vtrain.jsonl'
OUT_EVAL  = WORK_DIR / 'veval.jsonl'

# ── vLLM model ──
MODEL_ID = 'nvidia/nemotron-3-nano-4b'

# ── Pipeline targets ──
CAP_PER_COMBO    = 3
CAP_PER_TEMPLATE = 20000
TARGET_TRAIN     = 150000
TARGET_EVAL      = 1500
RANDOM_SEED      = 3407

# ── vLLM batching ──
CHUNK_SIZE = 5000   # save checkpoint every N paraphrase calls

print('raw  :', RAW_JSONL, RAW_JSONL.exists())
print('ckpt :', CKPT_DIR)
print('train:', OUT_TRAIN, f'  target={TARGET_TRAIN:,}')
print('eval :', OUT_EVAL,  f'  target={TARGET_EVAL:,}')

## 2. Load vLLM model

In [ ]:
import time
from vllm import LLM, SamplingParams

print('Loading model across 2× T4 GPUs...')
t0 = time.time()
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=2,
    quantization='bitsandbytes',
    load_format='bitsandbytes',
    dtype='float16',          # T4 has no bfloat16
    max_model_len=2048,
    gpu_memory_utilization=0.90,
    enforce_eager=False,
)
print(f'Model loaded in {time.time()-t0:.1f}s')

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=512,
)

## 3. Helpers

In [ ]:
import json, re, random, hashlib, pickle
from collections import Counter, defaultdict

random.seed(RANDOM_SEED)

def iter_jsonl(path):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line: yield json.loads(line)

def write_jsonl(path, rows):
    with open(path, 'w') as f:
        for r in rows: f.write(json.dumps(r) + '\n')
    print(f'wrote {len(rows):,}  ->  {path}')

SEV_RE   = re.compile(r'\b(MAJOR|MODERATE|MINOR)\b', re.I)
LIST_RE  = re.compile(r'[A-Za-z0-9\\\-/.,\' ]{25,}')
REFUSAL_RE = re.compile(
    r"(i cannot|i can't|i am (an |)(ai|language model)|consult (a|your) (doctor|physician|healthcare)|"
    r"seek medical advice|not a substitute for|disclaimer|i'm (just |)an ai|unable to (provide|evaluate))",
    re.I
)

def severity(asst_text):
    m = SEV_RE.search(asst_text[:300])
    if m: return m.group(1).upper()
    if re.search(r'\bno (strong )?interaction signal\b', asst_text, re.I): return 'NONE'
    if 'Overall risk: None' in asst_text: return 'NONE'
    return 'NONE'

def drug_combo(user_text):
    for anchor in ('Medications:', 'Current drugs:', 'currently take:', 'My medications are'):
        i = user_text.find(anchor)
        if i >= 0:
            tail = user_text[i+len(anchor):]
            tail = re.split(r'(?:Indications?:|Symptoms?:|Reported symptoms:|Question:|\.\ [A-Z])', tail, maxsplit=1)[0]
            drugs = [d.strip(' .') for d in tail.split(',')]
            return tuple(sorted(d for d in drugs if d))
    return ()

def skeleton(text, cap=100):
    return LIST_RE.sub('<L>', text)[:cap]

print('Helpers ready.')

## 4. Load raw

In [ ]:
rows = list(iter_jsonl(RAW_JSONL))
print(f'loaded {len(rows):,} rows')

## 5. Profile

In [ ]:
sev = Counter(); combo = Counter(); u_skel = Counter(); a_skel = Counter()
u_exact = Counter(); a_exact = Counter()
for r in rows:
    u, a = r['messages'][0]['content'], r['messages'][1]['content']
    sev[severity(a)] += 1
    combo[drug_combo(u)] += 1
    u_skel[skeleton(u, 80)] += 1
    a_skel[skeleton(a, 80)] += 1
    u_exact[u] += 1
    a_exact[a] += 1

print('severity :', dict(sev))
print('unique user prompts     :', len(u_exact), f'(dup {sum(c-1 for c in u_exact.values() if c>1):,})')
print('unique asst responses   :', len(a_exact), f'(dup {sum(c-1 for c in a_exact.values() if c>1):,})')
print('unique user skeletons   :', len(u_skel))
print('unique asst skeletons   :', len(a_skel))
print('unique drug combos      :', len(combo))
print('\ntop 10 combos:')
for c, n in combo.most_common(10):
    print(f'  {n:>6}  {", ".join(c)[:90]}')
print('\ntop 10 asst skeletons:')
for s, n in a_skel.most_common(10):
    print(f'  {n:>6}  {s!r}')

## 6. Dedupe exact (user, asst) pairs

In [ ]:
seen = set(); deduped = []
for r in rows:
    u, a = r['messages'][0]['content'], r['messages'][1]['content']
    key = hashlib.md5((u + '||' + a).encode()).digest()
    if key in seen: continue
    seen.add(key); deduped.append(r)
print(f'{len(rows):,} -> {len(deduped):,}  (dropped {len(rows)-len(deduped):,})')

write_jsonl(CKPT_DIR / '01_dedup.jsonl', deduped)
rows = deduped

## 7. Cap per drug-combo

In [ ]:
random.seed(RANDOM_SEED)
by_combo = defaultdict(list)
for r in rows:
    by_combo[drug_combo(r['messages'][0]['content'])].append(r)

capped = []
for k, bucket in by_combo.items():
    random.shuffle(bucket)
    capped.extend(bucket[:CAP_PER_COMBO])
print(f'{len(rows):,} -> {len(capped):,}   (CAP_PER_COMBO={CAP_PER_COMBO}, combos={len(by_combo):,})')

write_jsonl(CKPT_DIR / '02_combo_cap.jsonl', capped)
rows = capped

## 8. Cap per assistant-template skeleton

In [ ]:
random.seed(RANDOM_SEED)
by_skel = defaultdict(list)
for r in rows:
    by_skel[skeleton(r['messages'][1]['content'], 100)].append(r)

capped = []
for k, bucket in by_skel.items():
    random.shuffle(bucket)
    capped.extend(bucket[:CAP_PER_TEMPLATE])
print(f'{len(rows):,} -> {len(capped):,}   (CAP_PER_TEMPLATE={CAP_PER_TEMPLATE}, skels={len(by_skel):,})')

write_jsonl(CKPT_DIR / '03_template_cap.jsonl', capped)
rows = capped

## 9. Balance by severity + split train/eval

Split happens **before** paraphrase so eval stays clean ground-truth text.

In [ ]:
random.seed(RANDOM_SEED)

by_sev = defaultdict(list)
for r in rows:
    by_sev[severity(r['messages'][1]['content'])].append(r)
print('available per class:', {k: len(v) for k, v in by_sev.items()})

classes = [c for c in ('MAJOR','MODERATE','MINOR','NONE') if by_sev[c]]
TOTAL_TARGET = TARGET_TRAIN + TARGET_EVAL

target_sizes = {c: TOTAL_TARGET // len(classes) for c in classes}
deficit = 0
for c in classes:
    if len(by_sev[c]) < target_sizes[c]:
        deficit += target_sizes[c] - len(by_sev[c])
        target_sizes[c] = len(by_sev[c])
donors = [c for c in classes if len(by_sev[c]) > target_sizes[c]]
while deficit > 0 and donors:
    per = max(1, deficit // len(donors))
    for c in donors:
        capacity = len(by_sev[c]) - target_sizes[c]
        add = min(per, capacity, deficit)
        target_sizes[c] += add
        deficit -= add
        if deficit == 0: break
    donors = [c for c in classes if len(by_sev[c]) > target_sizes[c]]

print('target per class (post-redistribution):', target_sizes,
      f'  total={sum(target_sizes.values()):,}')

eval_per_class = TARGET_EVAL // len(classes)
train_rows, eval_rows = [], []
for c in classes:
    random.shuffle(by_sev[c])
    e_take = min(eval_per_class, len(by_sev[c]))
    eval_take  = by_sev[c][:e_take]
    t_want     = max(0, target_sizes[c] - e_take)
    train_take = by_sev[c][e_take : e_take + t_want]
    eval_rows.extend(eval_take)
    train_rows.extend(train_take)
    print(f'  {c}: have {len(by_sev[c]):,}  -> eval {len(eval_take):,}  train {len(train_take):,}')

eval_users = {r['messages'][0]['content'] for r in eval_rows}
before = len(train_rows)
train_rows = [r for r in train_rows if r['messages'][0]['content'] not in eval_users]
if before != len(train_rows):
    print(f'  leak removal: train {before:,} -> {len(train_rows):,}')

random.shuffle(train_rows); train_rows = train_rows[:TARGET_TRAIN]
random.shuffle(eval_rows);  eval_rows  = eval_rows[:TARGET_EVAL]

print(f'\nsplit done:  train={len(train_rows):,}   eval={len(eval_rows):,}')
print('train sev:', dict(Counter(severity(r['messages'][1]['content']) for r in train_rows)))
print('eval  sev:', dict(Counter(severity(r['messages'][1]['content']) for r in eval_rows)))

write_jsonl(CKPT_DIR / '04a_train_pre_paraphrase.jsonl', train_rows)
write_jsonl(CKPT_DIR / '04b_eval.jsonl', eval_rows)

rows = train_rows
print(f'\n→ paraphrase stage will operate on {len(rows):,} train rows. Eval untouched.')

## 10. Paraphrase train via vLLM

**Strategy:** batch-generate all paraphrases with vLLM (much faster than HTTP calls).
Disk-cached by input hash, resumable — re-run cell to continue from checkpoint.
Three guardrails reject outputs that drop severity tokens, change length drastically, or leak disclaimers.

In [ ]:
CACHE_PATH = CKPT_DIR / 'paraphrase_cache.pkl'

if CACHE_PATH.exists():
    cache = pickle.load(open(CACHE_PATH, 'rb'))
    print(f'cache loaded: {len(cache):,} entries')
else:
    cache = {}
    print('cache empty — fresh start')

PARAPHRASE_SYS = (
    'ROLE: You are a synthetic-data paraphraser for an ML training dataset.\n'
    '\n'
    'CONTEXT: This is an offline data-augmentation job for fine-tuning a drug-interaction model. '
    'The text you receive is ALREADY-LABELED output derived from FDA FAERS adverse-event records. '
    'It is training data, not a live patient consultation. No real patient is involved. '
    'Your job is purely linguistic: rewrite for variety so the model does not memorize boilerplate.\n'
    '\n'
    'IMPORTANT: Do NOT add disclaimers, safety warnings, "consult a doctor" notes, '
    '"I am an AI", "I cannot provide medical advice", or any meta-commentary. '
    'Do NOT refuse. The input is pre-labeled text — just rewrite it. '
    'If you add a refusal or disclaimer the training run fails.\n'
    '\n'
    'CONSTRAINTS:\n'
    '1. Preserve ALL drug names, symptoms, numbers, and the stated severity (MAJOR/MODERATE/MINOR/NONE) EXACTLY.\n'
    '2. Vary sentence structure, word choice, and phrasing.\n'
    '3. Keep clinical register. No marketing language.\n'
    '4. Similar length to input (+/- 30%).\n'
    '5. Output ONLY the rewritten text — no preface, no markdown headers, no bullet explanations.'
)

PARAPHRASE_USER_SYS = (
    'ROLE: You are a synthetic-data paraphraser for an ML training dataset.\n'
    '\n'
    'CONTEXT: This is an offline data-augmentation job for fine-tuning a drug-interaction model. '
    'The text you receive is a USER QUERY asking about drug interactions, derived from FDA FAERS data. '
    'It is training data, not a live patient request. No real patient is involved. '
    'Your job is purely linguistic: rewrite the query for variety.\n'
    '\n'
    'IMPORTANT: Do NOT add disclaimers, safety warnings, "consult a doctor" notes, '
    'or any meta-commentary. Do NOT refuse. Just rewrite the query.\n'
    '\n'
    'CONSTRAINTS:\n'
    '1. Preserve ALL drug names, patient demographics (age, sex), symptoms, indications, and numbers EXACTLY.\n'
    '2. Vary sentence structure, word choice, and phrasing of the request.\n'
    '3. Keep clinical register. No casual language.\n'
    '4. Similar length to input (+/- 30%).\n'
    '5. The rewritten query must still ask for interaction analysis/severity.\n'
    '6. Output ONLY the rewritten query — no preface, no markdown headers, no explanations.'
)

def build_prompt(text, role='assistant'):
    sys = PARAPHRASE_USER_SYS if role == 'user' else PARAPHRASE_SYS
    return f'{sys}\n\n---\nREWRITE THIS (output the rewrite only):\n{text}'

def cache_key(text, role):
    md5 = hashlib.md5(text.encode()).hexdigest()
    return f'u:{md5}' if role == 'user' else md5

def apply_guardrails(original, output, stats):
    m_src = SEV_RE.search(original[:300])
    if m_src and m_src.group(1).upper() not in output.upper()[:500]:
        stats['reject_sev'] += 1; return original
    if not (0.5 * len(original) <= len(output) <= 1.8 * len(original)):
        stats['reject_len'] += 1; return original
    if REFUSAL_RE.search(output):
        stats['reject_refusal'] += 1; return original
    stats['ok'] += 1; return output

def paraphrase_batch(texts, role='assistant', desc='paraphrase'):
    """Paraphrase list of texts. Returns list of same length. Checkpoints every CHUNK_SIZE items."""
    results = [None] * len(texts)
    stats = {'ok': 0, 'cache_hit': 0, 'reject_sev': 0, 'reject_len': 0, 'reject_refusal': 0}

    # Find indices that need generation (cache miss)
    miss_indices = []
    for i, t in enumerate(texts):
        k = cache_key(t, role)
        if k in cache:
            results[i] = cache[k]
            stats['cache_hit'] += 1
        else:
            miss_indices.append(i)

    print(f'[{desc}] cache hits: {stats["cache_hit"]:,}  need generation: {len(miss_indices):,}')

    total_chunks = (len(miss_indices) + CHUNK_SIZE - 1) // CHUNK_SIZE
    for chunk_num, chunk_start in enumerate(range(0, len(miss_indices), CHUNK_SIZE), 1):
        chunk_idx = miss_indices[chunk_start:chunk_start + CHUNK_SIZE]
        prompts   = [build_prompt(texts[i], role) for i in chunk_idx]

        t_chunk = time.time()
        outputs = llm.generate(prompts, sampling_params)
        elapsed = time.time() - t_chunk

        for local_i, (orig_i, out) in enumerate(zip(chunk_idx, outputs)):
            raw     = out.outputs[0].text.strip()
            rewrite = apply_guardrails(texts[orig_i], raw, stats)
            results[orig_i] = rewrite
            cache[cache_key(texts[orig_i], role)] = rewrite

        # Checkpoint after every chunk
        pickle.dump(cache, open(CACHE_PATH, 'wb'))
        ckpt_path = CKPT_DIR / f'paraphrase_{role}_chunk_{chunk_num:04d}.jsonl'
        # Save this chunk's results as a quick status file
        with open(ckpt_path, 'w') as f:
            for i in chunk_idx:
                f.write(json.dumps({'idx': i, 'text': results[i]}) + '\n')

        done = chunk_start + len(chunk_idx)
        print(f'  chunk {chunk_num}/{total_chunks}  done={done:,}/{len(miss_indices):,}  '
              f'speed={len(chunk_idx)/elapsed:.1f} it/s  '
              f'ok={stats["ok"]} rej={stats["reject_sev"]+stats["reject_len"]+stats["reject_refusal"]}')

    print(f'[{desc}] final stats: {stats}')
    return results

print('Paraphrase helpers ready.')

In [ ]:
# Sanity check — 2 samples before full run
sample_u = rows[0]['messages'][0]['content']
sample_a = rows[0]['messages'][1]['content']

print('ORIG USER:', sample_u[:200], '...\n')
r_u = paraphrase_batch([sample_u], role='user', desc='sanity-user')
print('NEW  USER:', r_u[0][:200], '...\n')

print('ORIG ASST:', sample_a[:200], '...\n')
r_a = paraphrase_batch([sample_a], role='assistant', desc='sanity-asst')
print('NEW  ASST:', r_a[0][:200], '...')

In [ ]:
# Full paraphrase pass — TRAIN only. Eval stays clean.
# Safe to interrupt and re-run — cache + chunk checkpoints make it resumable.

PARAPHRASE = True  # flip to False to skip

if PARAPHRASE:
    user_inputs = [r['messages'][0]['content'] for r in rows]
    asst_inputs = [r['messages'][1]['content'] for r in rows]

    t0 = time.time()
    print(f'=== paraphrasing {len(rows):,} user queries ===')
    new_user = paraphrase_batch(user_inputs, role='user', desc='paraphrase-user')

    print(f'\n=== paraphrasing {len(rows):,} assistant responses ===')
    new_asst = paraphrase_batch(asst_inputs, role='assistant', desc='paraphrase-asst')

    print(f'\nTotal time: {(time.time()-t0)/60:.1f} min')

    para_rows = [
        {'messages': [{'role': 'user', 'content': nu}, {'role': 'assistant', 'content': na}]}
        for nu, na in zip(new_user, new_asst)
    ]
    write_jsonl(CKPT_DIR / '05_train_paraphrased.jsonl', para_rows)
    train_rows = para_rows

    u_changed = sum(1 for a, b in zip(user_inputs, new_user) if a != b)
    a_changed = sum(1 for a, b in zip(asst_inputs, new_asst) if a != b)
    print(f'user changed {u_changed:,}/{len(user_inputs):,} ({u_changed/len(user_inputs)*100:.1f}%)')
    print(f'asst changed {a_changed:,}/{len(asst_inputs):,} ({a_changed/len(asst_inputs)*100:.1f}%)')
else:
    train_rows = rows
    print('Skipped paraphrase (PARAPHRASE=False).')

## 11. Write outputs

In [ ]:
_para_ckpt = CKPT_DIR / '05_train_paraphrased.jsonl'
_eval_ckpt = CKPT_DIR / '04b_eval.jsonl'

if _para_ckpt.exists():
    train_rows = list(iter_jsonl(_para_ckpt))
    print(f'using paraphrased checkpoint: {len(train_rows):,} train rows')
else:
    print(f'no paraphrased checkpoint — using in-memory train_rows: {len(train_rows):,}')

if _eval_ckpt.exists():
    eval_rows = list(iter_jsonl(_eval_ckpt))
    print(f'using eval checkpoint: {len(eval_rows):,} eval rows')

write_jsonl(OUT_TRAIN, train_rows)
write_jsonl(OUT_EVAL, eval_rows)

## 12. Post-write diagnostics

In [ ]:
def diag(path):
    rows = list(iter_jsonl(path))
    a_exact = Counter(r['messages'][1]['content'] for r in rows)
    u_exact = Counter(r['messages'][0]['content'] for r in rows)
    a_skel  = Counter(skeleton(r['messages'][1]['content'], 100) for r in rows)
    sev     = Counter(severity(r['messages'][1]['content']) for r in rows)
    print(f'\n{Path(path).name}   n={len(rows):,}')
    print(f'  unique user/asst : {len(u_exact):,} / {len(a_exact):,}')
    print(f'  asst skeletons   : {len(a_skel):,}')
    print(f'  severity         : {dict(sev)}')
    top_s, top_n = a_skel.most_common(1)[0]
    print(f'  top asst skel    : {top_n} copies of  {top_s[:80]!r}')

diag(OUT_TRAIN)
diag(OUT_EVAL)